In [33]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tokenizers import ByteLevelBPETokenizer


PARAMS = {}
PARAMS['vocab_file'] = '/root/model/pretrain/vocab.json'
PARAMS['merges_file'] = '/root/model/pretrain/merges.txt'
PARAMS['lowercase'] = True
PARAMS['add_prefix_space'] = True
TOKENIZER = ByteLevelBPETokenizer(**PARAMS)
MAX_LEN = 200

In [48]:
def jaccard(str1, str2):
    try:
        a = set(str1.lower().split()) 
        b = set(str2.lower().split())
        c = a.intersection(b)
        return float(len(c)) / (len(a) + len(b) - len(c))
    except:
        return -1.

def selectedText(text, start_idx, end_idx, offsets):
    selected_text = ""
    for ix in range(start_idx, end_idx + 1):
        selected_text += text[offsets[ix][0]: offsets[ix][1]]
        if (ix + 1) < len(offsets) and offsets[ix][1] < offsets[ix + 1][0]:
            selected_text += " "
    return selected_text

def getOffsets(text, sentiment):
    text = text.lower()
    sentiment = sentiment.lower().strip()
    text = " " + " ".join(text.lower().split())
    encodes = TOKENIZER.encode(text)
    sentiment = TOKENIZER.encode(sentiment).ids
    tokens = [0] + sentiment + [2, 2] + encodes.ids + [2]
    offsets = [(0, 0)] * 4 + encodes.offsets + [(0, 0)]
    padding = MAX_LEN - len(tokens)
    if padding > 0:
        tokens += [1] * padding
        offsets += [(0, 0)] * padding
    return offsets

def computeScore(text, sentiment, start_idx, end_idx, start_pred, end_pred):
    offsets = getOffsets(text, sentiment)
    if start_pred > end_pred:
        pred = text
    else:
        pred = selectedText(text, start_pred, end_pred, offsets)
    true = selectedText(text, start_idx, end_idx, offsets)
    return jaccard(true, pred)

In [49]:
valid = pd.read_csv('../../data/process/train.csv')
valid['text'] = valid['text'].fillna(' ')
valid = valid[valid['text'] != ' ']
valid = valid[valid['subset'] == 0]
valid = valid.reset_index(drop=True)[['text', 'sentiment']]
score = pd.read_csv('../../model/test_0.csv')
data = valid.join(score, how='inner')

In [50]:
data.head()

,text,sentiment,start_idx,end_idx,start_pred,end_pred
0,is back home now gonna miss every one,negative,8,8,9,9
1,Hes just not that into you,neutral,4,9,4,9
2,Playing Ghost Online is really interesting. Th...,positive,9,10,8,10
3,i`m soooooo sleeeeepy!!! the last day o` schoo...,negative,7,15,24,28
4,"Car not happy, big big dent in boot! Hoping t...",neutral,4,27,4,27


In [51]:
data['score'] = data.apply(lambda x : computeScore(*x), axis=1)

In [52]:
data.score.mean()

0.7063878235441238

In [53]:
data.head()

,text,sentiment,start_idx,end_idx,start_pred,end_pred,score
0,is back home now gonna miss every one,negative,8,8,9,9,0.0
1,Hes just not that into you,neutral,4,9,4,9,1.0
2,Playing Ghost Online is really interesting. Th...,positive,9,10,8,10,0.5
3,i`m soooooo sleeeeepy!!! the last day o` schoo...,negative,7,15,24,28,0.0
4,"Car not happy, big big dent in boot! Hoping t...",neutral,4,27,4,27,1.0
